Load Data

In [3]:
import pandas as pd
import numpy as np
trainFile = "/Users/matilderibeiro/Desktop/NOVA/Machine Learning/ML_Grupo18/data/train.csv"
df = pd.read_csv(trainFile)

In [2]:
print("Dataset Dimention (Columns, Rows):", df.shape)

Dataset Dimention (Columns, Rows): (6534, 39)


In [4]:
print("\n--- Datatype Information ---")
print(df.info())


--- Datatype Information ---
<class 'pandas.DataFrame'>
RangeIndex: 6534 entries, 0 to 6533
Data columns (total 39 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Total a Pagar                   6469 non-null   float64
 1   Total Bruto                     6469 non-null   float64
 2   Total Liquido                   6469 non-null   float64
 3   Total Imp_                      6469 non-null   float64
 4   Total Portes                    6469 non-null   float64
 5   Total Desc_ Global              6469 non-null   float64
 6   Total Desc_ Linha               6469 non-null   float64
 7   Custo                           6469 non-null   float64
 8   N_ Detalhes                     6469 non-null   float64
 9   invoice_count_current_purchase  6469 non-null   float64
 10  repeat_purchase_90d             6534 non-null   int64  
 11  days_since_previous_purchase    5673 non-null   float64
 12  customer_tenure

In [5]:
print("\n--- NA Values per column ---")
print(df.isnull().sum())


--- NA Values per column ---
Total a Pagar                       65
Total Bruto                         65
Total Liquido                       65
Total Imp_                          65
Total Portes                        65
Total Desc_ Global                  65
Total Desc_ Linha                   65
Custo                               65
N_ Detalhes                         65
invoice_count_current_purchase      65
repeat_purchase_90d                  0
days_since_previous_purchase       861
customer_tenure_days                65
prior_purchase_count                65
prior_total_spend                   65
prior_mean_purchase_value          862
prior_std_purchase_value          1113
prior_min_purchase_value           668
prior_max_purchase_value           668
prior_mean_gap_days               1308
prior_std_gap_days                1478
purchase_count_30d                  65
spend_30d                           65
purchase_count_90d                  65
spend_90d                         

Eliminar linhas —> 65 (representam 1% amostra): nestas quase rodas as variáveis da transação estão totalmente NaN

In [6]:
df_clean = df.dropna(subset=['Total a Pagar']).copy()

In [7]:
# 1. Verificar diretamente se ainda existem valores nulos nessa coluna (deve retornar 0)
print("NA Values in 'Total a Pagar':", df_clean['Total a Pagar'].isnull().sum())

NA Values in 'Total a Pagar': 0


In [8]:
print(f"Rows before: {len(df)} | Rows after: {len(df_clean)} | Removed rows: {len(df) - len(df_clean)}")

Rows before: 6534 | Rows after: 6469 | Removed rows: 65


Tratamento de Valores Ausentes Estruturais (MNAR)

Valores ausentes estruturais -> A ausencia de dados em variáveis de histórico de compras não é aleatória (Missing Not At Random - MNAR): devem-se a Novos Clientes / Primeira Compra: As variáveis days_since_previous_purchase, prior_mean_purchase_value, prior_mean_gap_days e prior_std_gap_days têm taxas elevadas de nulos (ex.: 861 a 1478 falhas). Clientes que efetuaram a sua primeira compra ou que têm apenas 1 compra anterior não possuem intervalo de tempo (gap) nem desvio-padrão (std) para calcular.

In [9]:
#Criar uma flag binária para novos clientes (Primeira Compra)
df_clean['is_first_purchase'] = df_clean['days_since_previous_purchase'].isnull().astype(int)

In [10]:
#Imputação por 0 para colunas de variação/desvio/médias que dependem de múltiplas compras
#Se o cliente só tem 1 compra anterior ou nenhuma, o desvio/intervalo é 0
zero_impute_cols = [
    'prior_std_purchase_value', 
    'prior_std_gap_days', 
    'prior_mean_gap_days'
]
df_clean[zero_impute_cols] = df_clean[zero_impute_cols].fillna(0)

In [11]:
#Imputação por valor indicador (-1) para contagens de tempo e médias de novos clientes
#O valor -1 assinala explicitamente ao algoritmo que não existe histórico anterior
indicator_impute_cols = [
    'days_since_previous_purchase',
    'prior_mean_purchase_value',
    'prior_min_purchase_value',
    'prior_max_purchase_value'
]
df_clean[indicator_impute_cols] = df_clean[indicator_impute_cols].fillna(-1)

In [12]:
#Validação da Flag de Novos Clientes
novos_clientes = df_clean['is_first_purchase'].sum()
print(f"[2] Flag 'is_first_purchase': {novos_clientes} novos clientes identificados ({(novos_clientes/len(df_clean))*100:.1f}%)")

[2] Flag 'is_first_purchase': 796 novos clientes identificados (12.3%)


In [14]:
# Validação das Imputações por -1
contagem_menos_um = (df_clean[indicator_impute_cols] == -1).sum().to_dict()
print(f"-> Valoração com -1 aplicada por coluna: {contagem_menos_um}")

-> Valoração com -1 aplicada por coluna: {'days_since_previous_purchase': 796, 'prior_mean_purchase_value': 797, 'prior_min_purchase_value': 603, 'prior_max_purchase_value': 603}


In [15]:
#Variáveis Temporais:
#purchase_date está registada como texto (str). Temos de convertê-la para datetime (pd.to_datetime) 
#para extrair atributos temporais úteis (ano, mês, dia da semana, trimestre)
# Converter para datetime
df_clean['purchase_date'] = pd.to_datetime(df_clean['purchase_date'])

O comportamento de compra dos clientes varia drasticamente dependendo do momento no tempo. Ao descompor a data, permites ao modelo aprender padrões específicos:

Sazonalidade e Meses (purchase_month / purchase_quarter):

Existem meses de pico de vendas (ex.: Black Friday em novembro, compras de Natal em dezembro ou épocas de férias em agosto).

Exemplo real no teu dataset: Nos dados do projeto, compras efetuadas em maio apresentam uma taxa de recompra a 90 dias de 72.4%, enquanto em março caem para 43.8%.

Comportamento Semanal (purchase_dayofweek / is_weekend):

O perfil de compras B2B (empresas) ou B2C (consumidores finais) muda entre dias úteis e fins de semana.

Exemplo real no teu dataset: As compras efetuadas em dias úteis (is_weekend=0) têm uma probabilidade de recompra de 64.4%, enquanto nos fins de semana (is_weekend=1) caem para 51.5%.

In [16]:
# Extração de atributos temporais relevantes
df_clean['purchase_year'] = df_clean['purchase_date'].dt.year
df_clean['purchase_month'] = df_clean['purchase_date'].dt.month
df_clean['purchase_day'] = df_clean['purchase_date'].dt.day
df_clean['purchase_dayofweek'] = df_clean['purchase_date'].dt.dayofweek  # 0=Segunda, 6=Domingo
df_clean['purchase_quarter'] = df_clean['purchase_date'].dt.quarter
df_clean['is_weekend'] = df_clean['purchase_dayofweek'].isin([5, 6]).astype(int)

Este bloco de código prepara as variáveis categóricas (colunas que contêm texto em vez de números) para poderem ser processadas pelos modelos de Machine Learning.

O que cada linha faz:

cat_cols = [...] -> Cria uma lista com os nomes de todas as colunas do dataset que são categóricas/texto.

df_clean[cat_cols] = df_clean[cat_cols].fillna('Missing') -> Procura por valores em falta (NaN / nulos) dentro dessas 6 colunas e substitui-os pela palavra 'Missing'.

Por que é que isto é fundamental?

⚬ Evita erros na codificação

⚬ Trata a ausência de informação como um padrão

⚬ Cria uma categoria explícita

In [17]:
#Codificação de Variáveis Categóricas
# Tratamento de NAs nas categóricas antes da codificação (atribuindo a classe 'Missing')
cat_cols = ['payment_type', 'salesperson', 'commercial_zone', 'transport_method', 'warehouse', 'document_series']
df_clean[cat_cols] = df_clean[cat_cols].fillna('Missing')

O que cada linha faz:

low_cardinality_cols = [...] -> Define a lista das colunas com poucas categorias únicas.

pd.get_dummies(...) -> Função do Pandas para transformar colunas de texto em binárias (0 ou 1).

drop_first=True (Muito Importante) -> Elimina a primeira categoria de cada coluna para evitar a 'Dummy Variable Trap' (multicolinearidade).

dtype=int -> Garante que o resultado das novas colunas seja em números inteiros (0 e 1).

In [19]:
# Baixa Cardinalidade -> One-Hot Encoding
# (payment_type, salesperson, commercial_zone, warehouse, document_series)
low_cardinality_cols = ['payment_type', 'salesperson', 'commercial_zone', 'warehouse', 'document_series']

df_clean = pd.get_dummies(
    df_clean, 
    columns=low_cardinality_cols, 
    drop_first=True,  # Evita a armadilha das variáveis dummy (multicolinearidade)
    dtype=int
)

In [20]:
# Alta Cardinalidade -> Frequency Encoding (ex.: transport_method com 80 categorias)
# Substitui cada categoria pela frequência proporcional/absoluta com que aparece
freq_map = df_clean['transport_method'].value_counts(normalize=True).to_dict()
df_clean['transport_method_freq'] = df_clean['transport_method'].map(freq_map)

A remoção dessas duas colunas específicas no final da fase de pré-processamento responde a duas razões técnicas essenciais:

purchase_date (Coluna Original da Data): Informação já foi extraída, incompatibilidade com o modelo, evitar redundância.

transport_method (Coluna Categórica Original): A categoria de texto foi substituída por um número via Frequency Encoding, evitando erros de treino com texto.

In [21]:
# Remover a coluna original de data e a categórica original de transport_method
df_clean = df_clean.drop(columns=['purchase_date', 'transport_method'])

In [22]:
# Criar a matriz de features (X) pronta para modelação (excluindo ID e Target)
features_to_exclude = ['ID', 'repeat_purchase_90d']
X = df_clean.drop(columns=features_to_exclude)
y = df_clean['repeat_purchase_90d']

In [23]:
# Validação Focada do Bloco
print("=== VALIDAÇÃO DA ENGENHARIA DE ATRIBUTOS E CATEGÓRICAS ===")
print(f"• Novos atributos temporais criados: ['purchase_year', 'purchase_month', 'purchase_day', 'purchase_dayofweek', 'purchase_quarter', 'is_weekend']")
print(f"• 'transport_method' codificado por frequência ('transport_method_freq')")
print(f"• Variáveis com One-Hot Encoding criadas com sucesso (total de colunas atuais: {df_clean.shape[1]})")
print(f"• Coluna 'ID' isolada e pronta para remoção no treino.")
print(f"• Coluna 'random_noise' mantida explicitamente para Feature Selection.")

=== VALIDAÇÃO DA ENGENHARIA DE ATRIBUTOS E CATEGÓRICAS ===
• Novos atributos temporais criados: ['purchase_year', 'purchase_month', 'purchase_day', 'purchase_dayofweek', 'purchase_quarter', 'is_weekend']
• 'transport_method' codificado por frequência ('transport_method_freq')
• Variáveis com One-Hot Encoding criadas com sucesso (total de colunas atuais: 66)
• Coluna 'ID' isolada e pronta para remoção no treino.
• Coluna 'random_noise' mantida explicitamente para Feature Selection.
